# 1. BDD vectoriel

Ce code est basé sur la méthodologie nvidia via le papier : 
- https://huggingface.co/blog/Albertmade/nvretriever-into-financial-text
- https://arxiv.org/pdf/2407.15831

### 1.1 Appel de la BDD

In [ ]:
import json
from tqdm import tqdm
from pymilvus import (
    MilvusClient, FieldSchema, CollectionSchema, DataType,
    Collection, connections, utility
)
from ollama import Client as OllamaClient

MILVUS_URI = "http://localhost:19530"
COLLECTION_NAME = "finetuning_v3"   # nouveau nom
EMBED_DIM = 1024
BATCH_SIZE = 128
JSON_PATH = "chunks.json"
MAX_TEXT = 60000

# -----------------------
# Embeddings (Ollama)
# -----------------------
import httpx
from typing import List

OLLAMA_URL = "http://localhost:11434"
EMB_MODEL = "qwen3-embedding:0.6b"

def emb_texts(texts: List[str]) -> List[List[float]]:
    # Ollama /api/embeddings (1 texte à la fois en API classique)
    # => on boucle, mais on garde un seul client HTTP
    out = []
    with httpx.Client(timeout=120.0) as client:
        for t in texts:
            r = client.post(
                f"{OLLAMA_URL}/api/embeddings",
                json={"model": EMB_MODEL, "prompt": t},
            )
            r.raise_for_status()
            out.append(r.json()["embedding"])
    return out

def emb_text(text: str) -> List[float]:
    return emb_texts([text])[0]


# -----------------------
# Connexion (pour Collection API)
# -----------------------
connections.connect(alias="default", uri=MILVUS_URI)

# Client (insert pratique)
client = MilvusClient(uri=MILVUS_URI)


### 1.2 Vérification des métadatas

In [ ]:
import pandas as pd

import pandas as pd
from pymilvus import Collection

def milvus_to_df(client, collection_name: str, batch_size: int = 10_000):
    # détecte les champs réellement présents dans le schéma
    col = Collection(collection_name)
    schema_fields = {f.name for f in col.schema.fields}

    wanted = [
        "id",
        "text",
        "source",
        "section_path",
        "section_title",
        "page_no",
        "chunk_type",
        "meta",
    ]
    output_fields = [f for f in wanted if f in schema_fields]

    if "id" not in output_fields or "text" not in output_fields:
        raise RuntimeError(
            f"Schéma incomplet pour le fine-tuning. Champs trouvés: {sorted(schema_fields)}"
        )

    offset = 0
    rows = []

    while True:
        batch = client.query(
            collection_name=collection_name,
            filter="",  # ou expr=""
            output_fields=output_fields,
            limit=batch_size,
            offset=offset,
        )
        if not batch:
            break

        rows.extend(batch)
        offset += len(batch)

    df = pd.DataFrame(rows)

    if "meta" in df.columns:
        meta_df = pd.json_normalize(df["meta"])
        df = pd.concat([df.drop(columns=["meta"]), meta_df], axis=1)

    cols_priority = [
        "id",
        "text",
        "source",
        "section_title",
        "section_path",
        "page_no",
        "chunk_type",
        "doc_id",
        "file_path",
        "chunk_index",
        "source_file",
        "type",
    ]
    cols_present = [c for c in cols_priority if c in df.columns]
    df = df[cols_present + [c for c in df.columns if c not in cols_present]]

    return df


# Définition de doc_id
df_kb = milvus_to_df(client, COLLECTION_NAME)
#df_kb["doc_id"] = df_kb["id"].str.split("::").str[0]     # chemin du pdf
df_kb["doc_id"] = df_kb["source"]
df_kb["chunk_id"] = df_kb["id"].str.split("::").str[-1] 
df_kb.head()


,id,text,source,section_title,section_path,page_no,chunk_type,doc_id,chunk_id
0,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...,## Docling: An Efficient Open-Source Toolkit f...,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...,Docling: An Efficient Open-Source Toolkit for ...,Docling: An Efficient Open-Source Toolkit for ...,1,noise,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...,c0_1_4
1,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...,"## 5 Performance\n\nIn this section, we charac...",/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...,5 Performance,5 Performance,4,noise,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...,c10_4_3
2,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...,## 5.1 Benchmark Dataset\n\nTo enable a meanin...,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...,5.1 Benchmark Dataset,5.1 Benchmark Dataset,4,noise,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...,c11_4_2
3,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...,## 5.2 System Configurations\n\nWe schedule ou...,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...,5.2 System Configurations,5.2 System Configurations,4,noise,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...,c12_4_4
4,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...,All experiments on the AWS EC2 VM are carried ...,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...,5.2 System Configurations,5.2 System Configurations,4,noise,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...,c12_4_8


In [3]:
from pymilvus import Collection

col = Collection(COLLECTION_NAME)
print([f.name for f in col.schema.fields])


['id', 'vector', 'text', 'source', 'doc_id', 'section_path', 'section_title', 'page_no', 'chunk_type', 'sparse']


In [4]:
len(df_kb)

531

#### 1.3. Insertion dans Milvus de doc_id si non présent

In [ ]:
import hashlib
from typing import Any, Dict, List, Sequence

from pymilvus import (
    MilvusClient,
    DataType,
    Function,
    FunctionType,
)


COLL = "finetuning"
BATCH_SIZE = 256
MAX_TEXT = 60000


JsonDict = Dict[str, Any]


def safe_primary_id(raw: str, *, max_len: int = 512) -> str:
    s = (raw or "").strip()
    if not s:
        return ""
    if len(s) <= max_len:
        return s
    h = hashlib.sha256(s.encode("utf-8")).hexdigest()
    keep = max_len - 1 - len(h)
    return (s[:keep] + "_" + h)[:max_len]


def ensure_hybrid_collection_with_doc_id(
    client: MilvusClient,
    *,
    name: str,
    dim: int,
    drop_if_exists: bool = True,
    analyzer_type: str = "english",
    consistency_level: str = "Strong",
    metric_dense: str = "COSINE",
):
    if client.has_collection(name):
        if drop_if_exists:
            try:
                client.release_collection(name)
            except Exception:
                pass
            client.drop_collection(name)
        else:
            client.load_collection(name)
            return

    schema = MilvusClient.create_schema(auto_id=False, enable_dynamic_field=True)

    schema.add_field("id", DataType.VARCHAR, is_primary=True, max_length=512)
    schema.add_field("vector", DataType.FLOAT_VECTOR, dim=dim)

    schema.add_field(
        "text",
        DataType.VARCHAR,
        max_length=65535,
        enable_analyzer=True,
        analyzer_params={"type": analyzer_type},
        enable_match=True,
    )

    schema.add_field("source", DataType.VARCHAR, max_length=1024)
    schema.add_field("doc_id", DataType.VARCHAR, max_length=1024)  # ✅ NEW
    schema.add_field("section_path", DataType.VARCHAR, max_length=2048)
    schema.add_field("section_title", DataType.VARCHAR, max_length=512)
    schema.add_field("page_no", DataType.INT64)
    schema.add_field("chunk_type", DataType.VARCHAR, max_length=64)

    # ✅ sparse + BM25 function
    schema.add_field("sparse", DataType.SPARSE_FLOAT_VECTOR)
    schema.add_function(
        Function(
            name="bm25",
            function_type=FunctionType.BM25,
            input_field_names=["text"],
            output_field_names="sparse",
        )
    )

    index_params = MilvusClient.prepare_index_params()
    index_params.add_index(
        field_name="vector",
        index_name="dense_index",
        index_type="HNSW",
        metric_type=metric_dense,
        params={"M": 16, "efConstruction": 200},
    )
    index_params.add_index(
        field_name="sparse",
        index_name="sparse_index",
        index_type="SPARSE_INVERTED_INDEX",
        metric_type="BM25",
    )

    client.create_collection(
        collection_name=name,
        schema=schema,
        index_params=index_params,
        consistency_level=consistency_level,
    )
    client.load_collection(name)


def insert_new_chunks(
    client: MilvusClient,
    items: Sequence[JsonDict],
    *,
    coll: str = COLL,
    default_source: str = "noise",
    default_chunk_type: str = "noise",
    batch_size: int = BATCH_SIZE,
    max_text: int = MAX_TEXT,
) -> int:
    batch: List[JsonDict] = []
    inserted = 0

    for obj in items:
        text = (obj.get("text") or "")[:max_text]
        if not text.strip():
            continue

        rid = safe_primary_id(obj.get("id") or "")
        if not rid:
            continue

        vec = emb_text(text)

        rec = {
            "id": rid,
            "vector": vec,
            "text": text,  # ✅ déclenche BM25 -> sparse via Function
            "source": obj.get("source", default_source) or default_source,
            "doc_id": obj.get("doc_id") or obj.get("source", default_source) or default_source,
            "section_path": obj.get("section_path", "") or "",
            "section_title": obj.get("section_title", "") or "",
            "page_no": int(obj.get("page_no", -1)) if obj.get("page_no") is not None else -1,
            "chunk_type": obj.get("chunk_type") or default_chunk_type,
        }
        batch.append(rec)

        if len(batch) >= batch_size:
            client.insert(collection_name=coll, data=batch)
            inserted += len(batch)
            batch.clear()

    if batch:
        client.insert(collection_name=coll, data=batch)
        inserted += len(batch)

    client.flush(collection_name=coll)
    return inserted


items = df_kb.to_dict(orient="records")




In [ ]:
NEW_COLL = COLLECTION_NAME + "_v3"

milvus = MilvusClient(uri=MILVUS_URI)

ensure_hybrid_collection_with_doc_id(milvus, name=NEW_COLL, dim=EMBED_DIM, drop_if_exists=True)

n = insert_new_chunks(milvus, items, coll=NEW_COLL, default_chunk_type="kb")
print("Inserted:", n)

Inserted: 531


In [17]:
from pymilvus import Collection

col = Collection(NEW_COLL)

print([f.name for f in col.schema.fields])

['id', 'vector', 'text', 'source', 'doc_id', 'section_path', 'section_title', 'page_no', 'chunk_type', 'sparse']


In [6]:
from pymilvus import Collection

col = Collection(COLLECTION_NAME)

print([f.name for f in col.schema.fields])

['id', 'vector', 'text', 'source', 'doc_id', 'section_path', 'section_title', 'page_no', 'chunk_type', 'sparse']


## 2. Construction du dataset pour le finetuning

### 2.1 simple negative

In [7]:
import random

def easy_negative_from_df(df, pos_doc_id: str, rng: random.Random) -> str:
    pool = df[df["doc_id"] != pos_doc_id]
    if pool.empty:
        pool = df
    row = pool.sample(1, random_state=rng.randint(0, 10**9)).iloc[0]
    return row["text"]


### 2.2 hard negative


- https://huggingface.co/blog/Albertmade/nvretriever-into-financial-text
- https://arxiv.org/pdf/2407.15831

In [8]:
from __future__ import annotations

import math
import random
import re
from dataclasses import dataclass
from typing import Optional, List, Dict, Any, Tuple

from pymilvus import Collection

_WORD_RE = re.compile(r"\w+", re.UNICODE)

def tokenize(s: str) -> set[str]:
    return set(_WORD_RE.findall((s or "").lower()))

def jaccard(a: str, b: str) -> float:
    A, B = tokenize(a), tokenize(b)
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)

def normalize_ws(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()

def milvus_quote(s: str) -> str:
    s = (s or "")
    s = s.replace("\\", "\\\\").replace('"', '\\"')
    s = s.replace("\n", " ").replace("\r", " ")
    return f'"{s}"'

def softmax(xs: List[float], temp: float = 1.0) -> List[float]:
    if not xs:
        return []
    t = max(temp, 1e-6)
    m = max(xs)
    exps = [math.exp((x - m) / t) for x in xs]
    Z = sum(exps) + 1e-12
    return [e / Z for e in exps]

def weighted_sample_without_replacement(
    items: List[Any],
    weights: List[float],
    k: int,
    rng: random.Random,
) -> List[Any]:
    # petit algo simple (suffisant ici)
    picked = []
    pool = list(items)
    w = list(weights)
    k = min(k, len(pool))
    for _ in range(k):
        s = sum(w) + 1e-12
        r = rng.random() * s
        acc = 0.0
        idx = 0
        for i, wi in enumerate(w):
            acc += wi
            if acc >= r:
                idx = i
                break
        picked.append(pool.pop(idx))
        w.pop(idx)
    return picked


@dataclass
class HardNegConfig:
    # Retrieval
    top_k_search: int = 200
    nprobe: int = 16

    # NV-Retriever positive-aware (TopK-PercPos)
    perc_pos: float = 0.95            # paper best (95%) :contentReference[oaicite:3]{index=3}
    use_shift: bool = True
    shift_n: int = 10                 # "Top-K shifted by N" (N=10) :contentReference[oaicite:4]{index=4}

    # Sampling
    sample_top_k: int = 10            # paper best: sample among top-10 :contentReference[oaicite:5]{index=5}
    n_negs: int = 4                   # paper often uses 4 hard negs :contentReference[oaicite:6]{index=6}
    sampling: str = "top1_plus_sampled"  # "sampled" | "top1_plus_sampled" | "topk"

    # Filters
    reject_if_score_above: float = 0.88
    reject_if_jaccard_pos_above: float = 0.35
    min_chars: int = 60

    # If Milvus returns distance instead of similarity
    score_is_distance: bool = False

    # Softmax temperature for sampling (lower = focus on hardest)
    softmax_temp: float = 1.0


class MilvusHardNegativeMinerNV:
    """
    NV-Retriever style miner:
    - compute pos_score (query vs pos chunk via id)
    - retrieve negatives candidates
    - filter with TopK-PercPos: neg_score < pos_score * perc_pos
    - optional shift by N (skip top N ranks)
    - pick negatives with top1 + sampled(top-k) using softmax on scores
    """

    def __init__(
        self,
        collection_name: str,
        anns_field: str,
        *,
        id_field: str = "id",
        doc_id_field: str = "doc_id",
        output_fields: Optional[List[str]] = None,
    ):
        self.col = Collection(collection_name)
        self.col.load()
        self.anns_field = anns_field
        self.id_field = id_field
        self.doc_id_field = doc_id_field
        self.output_fields = output_fields or [
            self.id_field, "text", self.doc_id_field, "section_path", "chunk_type"
        ]

    def _to_sim(self, score: float, score_is_distance: bool) -> float:
        return -score if score_is_distance else score

    def _search(self, qvec, *, cfg: HardNegConfig, expr: Optional[str], limit: int):
        kwargs: Dict[str, Any] = dict(
            data=qvec,
            anns_field=self.anns_field,
            param={"metric_type": "COSINE", "params": {"nprobe": cfg.nprobe}},
            limit=limit,
            output_fields=self.output_fields,
        )
        if expr:
            kwargs["expr"] = expr
        return self.col.search(**kwargs)

    def _pos_score_from_id(self, *, qvec, pos_chunk_id: Any, cfg: HardNegConfig) -> Optional[float]:
        if pos_chunk_id is None:
            return None
        if isinstance(pos_chunk_id, (int, float)) and not isinstance(pos_chunk_id, bool):
            expr = f"{self.id_field} == {int(pos_chunk_id)}"
        else:
            expr = f"{self.id_field} == {milvus_quote(str(pos_chunk_id))}"

        try:
            res = self._search(qvec, cfg=cfg, expr=expr, limit=1)
        except Exception:
            return None

        hits = res[0] if res else []
        if not hits:
            return None
        return self._to_sim(float(hits[0].score), cfg.score_is_distance)

    def mine_many(
        self,
        *,
        query_text: str,
        pos_text: str,
        pos_doc_id: str,
        pos_chunk_id: Any,
        emb_fn,
        rng: Optional[random.Random] = None,
        cfg: Optional[HardNegConfig] = None,
        pos_section_path: Optional[str] = None,
        pos_chunk_type: Optional[str] = None,
        allow_same_doc: bool = False,
    ) -> List[str]:
        rng = rng or random.Random()
        cfg = cfg or HardNegConfig()

        q = normalize_ws(query_text)
        if not q:
            return []

        qvec = [emb_fn(q)]

        # 1) pos_score anchor (paper’s “positive-aware” anchor) :contentReference[oaicite:7]{index=7}
        pos_score = self._pos_score_from_id(qvec=qvec, pos_chunk_id=pos_chunk_id, cfg=cfg)
        if pos_score is None:
            # sans pos_score, on peut fallback (mais NV-Retriever insiste sur pos-aware)
            return []

        # 2) expr Milvus sur champs stables
        expr_parts = []
        if not allow_same_doc and pos_doc_id:
            expr_parts.append(f"{self.doc_id_field} != {milvus_quote(str(pos_doc_id))}")
        if pos_chunk_type:
            expr_parts.append(f"chunk_type != {milvus_quote(str(pos_chunk_type))}")
        expr = " and ".join(expr_parts) if expr_parts else None

        # 3) search candidates
        try:
            res = self._search(qvec, cfg=cfg, expr=expr, limit=cfg.top_k_search)
        except Exception:
            return []

        hits = res[0] if res else []
        if not hits:
            return []

        pos_section_path = (pos_section_path or "").strip() or None

        # 4) filter candidates (TopK-PercPos + guards)
        # TopK-PercPos rule: neg_score < pos_score * 0.95 :contentReference[oaicite:8]{index=8}
        max_neg = pos_score * cfg.perc_pos

        filtered: List[Tuple[float, str]] = []  # (sim, text)
        for rank, hit in enumerate(hits):
            ent = hit.entity
            neg_text = normalize_ws(ent.get("text") or "")
            if not neg_text or len(neg_text) < cfg.min_chars:
                continue

            if pos_section_path:
                hit_path = (ent.get("section_path") or "").strip()
                if hit_path and hit_path == pos_section_path:
                    continue

            sim = self._to_sim(float(hit.score), cfg.score_is_distance)

            # absolute guard to drop very close (often false negs / duplicates)
            if sim >= cfg.reject_if_score_above:
                continue

            # pos-aware false-negative removal
            if sim >= max_neg:
                continue

            # near-duplicate removal vs positive
            if pos_text and jaccard(pos_text, neg_text) >= cfg.reject_if_jaccard_pos_above:
                continue

            filtered.append((sim, neg_text))

        if not filtered:
            return []

        # 5) shift-by-N option (ignore top-N ranks) :contentReference[oaicite:9]{index=9}
        if cfg.use_shift and len(filtered) > cfg.shift_n:
            filtered = filtered[cfg.shift_n:]

        if not filtered:
            return []

        # 6) choose N negatives (paper uses: sampled top-k or top1+sampled top-k) :contentReference[oaicite:10]{index=10}
        # sort by sim desc (hardest first)
        filtered.sort(key=lambda x: x[0], reverse=True)

        if cfg.sampling == "topk":
            return [t for _, t in filtered[: cfg.n_negs]]

        top_pool = filtered[: min(cfg.sample_top_k, len(filtered))]

        if cfg.sampling == "sampled":
            sims = [s for s, _ in top_pool]
            probs = softmax(sims, temp=cfg.softmax_temp)
            picks = weighted_sample_without_replacement(top_pool, probs, cfg.n_negs, rng)
            return [t for _, t in picks]

        # "top1_plus_sampled" (recommended)
        top1 = top_pool[0]
        rest = top_pool[1:]
        if not rest:
            return [top1[1]]

        sims = [s for s, _ in rest]
        probs = softmax(sims, temp=cfg.softmax_temp)
        picks = weighted_sample_without_replacement(rest, probs, max(cfg.n_negs - 1, 0), rng)
        out = [top1[1]] + [t for _, t in picks]
        return out


### 1.3 making triplets

In [9]:
##### 
import httpx, json, re
from typing import List, Dict, Any, Optional

def _clean_q(q: str) -> str:
    q = re.sub(r"\s+", " ", (q or "").strip())
    q = re.sub(r"^\s*[\-\•\*\d]+[\)\.\:\-]?\s*", "", q).strip()
    return q

def ollama_make_queries(
    chunk_text: str,
    *,
    n: int = 3,
    ollama_url: str = "http://localhost:11434",
    model: str = "qwen2.5:7b",   # ✅
    temperature: float = 0.2,
    timeout_s: float = 90.0,
    retries: int = 1,
) -> List[str]:
    chunk_text = (chunk_text or "").strip()
    if not chunk_text:
        return []
    chunk_text = chunk_text[:2200]

    schema: Dict[str, Any] = {
        "type": "object",
        "properties": {
            "queries": {
                "type": "array",
                "items": {"type": "string"},
                "minItems": n,
                "maxItems": n,
            }
        },
        "required": ["queries"],
        "additionalProperties": False,
    }

    prompt = f"""
Génère EXACTEMENT {n} requêtes de recherche en français pour retrouver l'extrait ci-dessous.

Contraintes STRICTES:
- 6 à 14 mots par requête
- pas d'introduction, pas de numérotation, pas de puces
- ne copie pas de phrases entières de l'extrait
- réponds UNIQUEMENT en JSON conforme au schéma

EXTRAIT:
{chunk_text}
""".strip()

    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "format": schema,   # ✅ sortie structurée
        "options": {"temperature": temperature, "num_predict": 220},
    }

    def call(client: httpx.Client, pay: Dict[str, Any]) -> str:
        r = client.post(f"{ollama_url}/api/generate", json=pay)
        r.raise_for_status()
        return (r.json().get("response") or "").strip()

    def parse(raw: str) -> List[str]:
        try:
            obj = json.loads(raw)
        except Exception:
            return []
        qs = obj.get("queries") or []
        out, seen = [], set()
        for q in qs:
            if not isinstance(q, str):
                continue
            q = _clean_q(q)
            if not q:
                continue
            k = q.lower()
            if k in seen:
                continue
            seen.add(k)
            out.append(q)
        return out

    with httpx.Client(timeout=timeout_s) as client:
        for _ in range(retries + 1):
            raw = call(client, payload)
            out = parse(raw)
            if len(out) == n:
                return out

            # 🔧 réparation (petit coût, gros gain)
            repair_payload = dict(payload)
            repair_payload["prompt"] = f"""
Corrige pour produire UNIQUEMENT un JSON valide conforme au schéma suivant, sans texte hors JSON.
Schéma: {json.dumps(schema, ensure_ascii=False)}

Sortie à corriger:
{raw}
""".strip()
            repair_payload["options"] = {"temperature": 0.0, "num_predict": 180}

            raw2 = call(client, repair_payload)
            out2 = parse(raw2)
            if len(out2) == n:
                return out2

    return []


In [11]:
######
import json
import random
from pathlib import Path
from tqdm import tqdm

def build_triplets_jsonl_nv(
    df_kb,
    *,
    collection_name: str,
    anns_field: str,
    out_path: str,
    queries_per_chunk: int = 2,
    hard_ratio: float = 0.9,     # plus haut (les easy random sont trop faciles)
    seed: int = 42,
    ollama_llm_model: str = "qwen2.5:7b",
):
    df = df_kb.copy()
    df = df[df["text"].astype(str).str.strip().ne("")].reset_index(drop=True)

    rng = random.Random(seed)

    miner = MilvusHardNegativeMinerNV(collection_name=collection_name, anns_field=anns_field)

    cfg = HardNegConfig(
        top_k_search=80,
        nprobe=16,
        perc_pos=0.92,            # plus strict (petit corpus)
        use_shift=True,
        shift_n=4,                # pas trop grand
        sample_top_k=10,          # OK
        n_negs=3,                 # 2-3 pour éviter répétitions
        sampling="top1_plus_sampled",
        reject_if_score_above=0.88,
        reject_if_jaccard_pos_above=0.30,  # petit corpus -> plus strict anti-dup
        min_chars=60,
        softmax_temp=0.7,         # focus plus sur les plus durs
    )


    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with out_path.open("w", encoding="utf-8") as f:
        for _, row in tqdm(df.iterrows(), total=len(df), desc="Triplets NV"):
            pos_text = str(row["text"] or "").strip()
            if not pos_text:
                continue

            pos_id = row["id"]
            pos_doc_id = str(row["doc_id"] or "").strip()

            pos_section_path = str(row.get("section_path") or "").strip() or None
            pos_chunk_type = str(row.get("chunk_type") or "").strip() or None

            title = str(row.get("section_title", "") or "").strip()
            prefix = f"TITRE: {title}\n\n" if title else ""

            queries = ollama_make_queries(
                prefix + pos_text,
                n=queries_per_chunk,
                model=ollama_llm_model,
            )
            if not queries:
                continue

            for q in queries:
                negs: List[str] = []
                if rng.random() < hard_ratio:
                    negs = miner.mine_many(
                        query_text=q,
                        pos_text=pos_text,
                        pos_doc_id=pos_doc_id,
                        pos_chunk_id=pos_id,
                        pos_section_path=pos_section_path,
                        pos_chunk_type=pos_chunk_type,
                        emb_fn=emb_text,
                        rng=rng,
                        cfg=cfg,
                        allow_same_doc=False,
                    )

                # fallback: si mining échoue, on fait 1 easy neg
                if not negs:
                    negs = [easy_negative_from_df(df, pos_doc_id, rng)]

                # écrit 1 ligne par négatif (TripletLoss)
                for neg in negs:
                    rec = {
                        "query": q,
                        "chunk_pos": pos_text,
                        "chunk_neg": neg,
                        "pos_id": pos_id,
                        "pos_doc_id": pos_doc_id,
                    }
                    f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    print(f"✅ Wrote -> {out_path}")


In [14]:
from pymilvus import Collection
col = Collection(COLLECTION_NAME) #NEW_COLL)

for f in col.schema.fields:
    print(f.name, f.dtype)


id 21
vector 101
text 21
source 21
doc_id 21
section_path 21
section_title 21
page_no 5
chunk_type 21
sparse 104


In [13]:
import random

def split_docs(df, seed=42):
    docs = df["doc_id"].unique().tolist()
    rng = random.Random(seed)
    rng.shuffle(docs)

    train_docs = set(docs[:3])
    valid_docs = set(docs[3:4])
    test_docs  = set(docs[4:5])

    return (
        df[df["doc_id"].isin(train_docs)].reset_index(drop=True),
        df[df["doc_id"].isin(valid_docs)].reset_index(drop=True),
        df[df["doc_id"].isin(test_docs)].reset_index(drop=True),
    )


In [15]:
ANNS_FIELD = "vector"


train_df, valid_df, test_df = split_docs(df_kb, seed=42)

build_triplets_jsonl_nv(
    train_df,
    collection_name=COLLECTION_NAME, #NEW_COLL,
    anns_field=ANNS_FIELD,
    out_path="dataset_triplets/train.jsonl",
    queries_per_chunk=2,
    hard_ratio=0.9,
    seed=42,
)

build_triplets_jsonl_nv(
    valid_df,
    collection_name=COLLECTION_NAME, #NEW_COLL,
    anns_field=ANNS_FIELD,
    out_path="dataset_triplets/valid.jsonl",
    queries_per_chunk=2,
    hard_ratio=0.9,
    seed=43,
)

build_triplets_jsonl_nv(
    test_df,
    collection_name=COLLECTION_NAME, #NEW_COLL,
    anns_field=ANNS_FIELD,
    out_path="dataset_triplets/test.jsonl",
    queries_per_chunk=2,
    hard_ratio=0.9,
    seed=44,
)


Triplets NV: 100%|██████████| 128/128 [14:49<00:00,  6.95s/it]


✅ Wrote -> dataset_triplets/train.jsonl


Triplets NV: 100%|██████████| 74/74 [07:43<00:00,  6.26s/it]


✅ Wrote -> dataset_triplets/valid.jsonl


Triplets NV: 100%|██████████| 46/46 [05:31<00:00,  7.20s/it]

✅ Wrote -> dataset_triplets/test.jsonl


In [54]:
import pandas as pd

train_triplets = pd.read_json(
    "dataset_triplets/train.jsonl",
    lines=True,
)

train_triplets.head()


,query,chunk_pos,chunk_neg,pos_id,pos_doc_id
0,survey context engineering large language models,† Also affiliated with: (1)Key Laboratory of N...,- B1 Large amounts of upfront investment into ...,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...
1,survey context engineering large language models,† Also affiliated with: (1)Key Laboratory of N...,## 4.3. Evaluation Results Evaluations on Stan...,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...
2,survey context engineering large language models,† Also affiliated with: (1)Key Laboratory of N...,- Microsoft Phi series. Phi-2 (2.7bn) achieves...,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...
3,context engineering llm agent multi-agent systems,† Also affiliated with: (1)Key Laboratory of N...,## We contend that SLMs are - V1 principally s...,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...
4,context engineering llm agent multi-agent systems,† Also affiliated with: (1)Key Laboratory of N...,## 3.6 Agentic systems are naturally heterogen...,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...


In [55]:
train_triplets['query'].iloc[1], train_triplets['query'].iloc[0], train_triplets['query'].iloc[2]

('survey context engineering large language models',
 'survey context engineering large language models',
 'survey context engineering large language models')

In [56]:
train_triplets['query'].iloc[2], train_triplets['chunk_pos'].iloc[2], train_triplets['chunk_neg'].iloc[2]

('survey context engineering large language models',
 '† Also affiliated with: (1)Key Laboratory of Network Data Science and Technology, ICT, CAS; (2)State Key Laboratory of AI Safety\n\nCorresponding Author\n\nKeywords : Context Engineering, Large Language Models, LLM Agent, Multi-Agent Systems\n\nDate\n\n: July 21, 2025\n\nCode Repository\n\n: https://github.com/Meirtz/Awesome-Context-Engineering\n\nContact\n\n: meilingrui25b@ict.ac.cn, liushenghua@ict.ac.cn',
 '- Microsoft Phi series. Phi-2 (2.7bn) achieves commonsense reasoning scores and code generation scores on par with 30bn models while running ∼ 15 × faster [34]. Phi-3 small (7bn) [3] achieves language understanding and commonsense reasoning on par with and code generation scores running up to 70bn models of the same generation. - NVIDIA Nemotron-H family. The 2/4.8/9bn hybrid Mamba-Transformer models achieve instruction following and code-generation accuracy comparable to dense 30bn LLMs of the same generation at an order-of-

In [57]:
train_triplets["qlen"] = train_triplets["query"].str.split().str.len()
train_triplets["query"].head(20), train_triplets["qlen"].describe()


(0      survey context engineering large language models
 1      survey context engineering large language models
 2      survey context engineering large language models
 3     context engineering llm agent multi-agent systems
 4     context engineering llm agent multi-agent systems
 5     context engineering llm agent multi-agent systems
 6      survey context engineering large language models
 7      survey context engineering large language models
 8      survey context engineering large language models
 9     authors survey context engineering large langu...
 10    authors survey context engineering large langu...
 11    authors survey context engineering large langu...
 12    context engineering large language models univ...
 13    context engineering large language models univ...
 14    context engineering large language models univ...
 15    survey retrieval augmented generation rag memo...
 16    survey retrieval augmented generation rag memo...
 17    survey retrieval augment

## 2. Finetuning 

In [1]:
#!pip install -U sentence-transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.3 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [43]:
valid = pd.read_json("dataset_triplets/valid.jsonl", lines=True)
valid.head()

,query,chunk_pos,chunk_neg,pos_id,pos_doc_id
0,Top-K Sampling pour les tokens les plus probables,## Top-K Sampling:\n\n- Def: Picks top k most ...,<!-- image:#/pictures/2 --> > **Résumé image**...,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...,1743034163900.pdf
1,Exemple de Top-K Sampling avec un petit jeu de...,## Top-K Sampling:\n\n- Def: Picks top k most ...,## * Characteristics * :\n\n- Logits are * unn...,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...,1743034163900.pdf
2,Top-P sampling definition,## Top-P (Nucleus) Sampling:\n\n- Def: Picks t...,## What actually DPO Does? =\n\n1. DPO fine-tu...,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...,1743034163900.pdf
3,Nucleus sampling algorithm,## Top-P (Nucleus) Sampling:\n\n- Def: Picks t...,across long conversations. For document analys...,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...,1743034163900.pdf
4,Extrait sur le modèle pirate My Pirate Model,## My Pirate Model:\n\n- Issue: Extra [INST] p...,<!-- image:#/pictures/2 --> > **Résumé image**...,/var/folders/bj/77dxlmzs5qg8p7gcm57r_4f40000gn...,1743034163900.pdf


In [45]:
# from sentence_transformers import SentenceTransformer

# from huggingface_hub import login
# login()

# MODEL_ID = "Fe2x/finetuned-embedding-rag"
# model = SentenceTransformer(MODEL_ID)

In [ ]:
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers import SentenceTransformer
import numpy as np

def build_ir_eval_from_triplets(triplets):
    queries = {}
    corpus = {}
    relevant_docs = {}

    for i, t in enumerate(triplets):
        qid = str(i)
        pos_id = str(t["pos_id"])

        queries[qid] = t["query"]
        corpus[pos_id] = t["chunk_pos"]
        relevant_docs[qid] = {pos_id}

        # optionnel: ajouter aussi les negatives au corpus pour rendre l’éval plus réaliste
        neg_id = f"neg_{i}"
        corpus[neg_id] = t["chunk_neg"]

    return queries, corpus, relevant_docs

queries, corpus, relevant_docs = build_ir_eval_from_triplets(valid)

evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="valid_ir",
    show_progress_bar=True,
    k_values=[1, 3, 5, 10, 20],
)

print("Base IR:")
evaluator(base)

print("FT IR:")
evaluator(model)



FileNotFoundError: Unable to find '/content/dataset_triplets/train.jsonl'

In [ ]:
import math
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.losses import TripletLoss
from torch.utils.data import DataLoader

BASE_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
OUT_DIR = "Fe2x/finetuned-embedder"

def to_input_examples(split):
    return [
        InputExample(texts=[r["query"], r["chunk_pos"], r["chunk_neg"]])
        for r in split
    ]

train_examples = to_input_examples(ds["train"])
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

model = SentenceTransformer(BASE_MODEL)
loss = TripletLoss(model=model)

epochs = 3
warmup_steps = math.ceil(len(train_dataloader) * epochs * 0.1)

model.fit(
    train_objectives=[(train_dataloader, loss)],
    epochs=epochs,
    warmup_steps=warmup_steps,
    output_path=OUT_DIR,
    show_progress_bar=True,
)

print("✅ Saved locally in:", OUT_DIR)


In [ ]:
import numpy as np
from sentence_transformers.util import cos_sim

def eval_triplet_accuracy(model, split, n=200):
    ok = 0
    total = min(n, len(split))
    for i in range(total):
        r = split[i]
        q = model.encode(r["query"], normalize_embeddings=True)
        p = model.encode(r["chunk_pos"], normalize_embeddings=True)
        neg = model.encode(r["chunk_neg"], normalize_embeddings=True)

        if float(np.dot(q, p)) > float(np.dot(q, neg)):
            ok += 1
    return ok / total

base = SentenceTransformer(BASE_MODEL)
ft = SentenceTransformer(OUT_DIR)

print("Base acc:", eval_triplet_accuracy(base, ds["validation"], n=200))
print("FT   acc:", eval_triplet_accuracy(ft, ds["validation"], n=200))
